# Create Dust_temp.in files
Store each file in its respective climate input directory on s3, which means it'll get copied over to the run directory. (means we need to have already created the climate input dir!)

## Read in dust flux timeseries by site 

In [5]:
import math
import os 
from typing import Optional, List

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr 

from ew_workflows import scepter_helperFxns as shf

In [24]:
# --- where to save outputs ----------
#
#  ----------------------------------------------------------------
# [ RUN NAMES ]
# exe_central1p5 ; exe_central2p0 ; exe_limBio1p5 ; exe_lowCCS1p5 ; 
# exe_lowCDR1p5 ; exe_noBiochar1p5 ; exe_noWeathering1p5 ; 
# exe_ref1p5 ; exe_techBehaviorChange1p5 ; exe_techBehaviorChange2p0 ;
# 
# (updated dust)
# exe_central1p5_max10tha   exe_noWeathering1p5_max5tha
# 
# [ RUNS TO COMPARE ]
# exe_central1p5 ; exe_noWeathering1p5 ; exe_techBehaviorChange1p5
# ----------------------------------------------------------------
#
#
# --- add crop area options (it will save multiple identical clim dirs,
#     but rock-application-specific files will be added later)
REPEAT_CLIM_FOR_CROP_AREA = False
crop_area_frac_arr = np.round(np.linspace(0.1, 1, 10), 2) # from 10% to 100% in 10% increments
# --- add application rate options 
REPEAT_CLIM_FOR_APP_RATE = True 
app_rate_arr = np.array([5, 7.5, 10, 15, 20, 30, 40, 50])  
# -------------------------------------------------------------------
#
#
# <Site>_<deg_resolution>_<climtype>
RUNNAME = "exe_techBehaviorChange1p5"
climtype = "yearly_ltm"
# ------------------------------------
LOCATION_DATASETS = {
    "largest_cropland": {
        "label": "largestCropland",
    },
    "wettest_crop25pct": {
        "label": "wettestSite_crop25pct",
    },
}

# --- select which location dataset to use
LOCATION_KEY = "largest_cropland"
loc = LOCATION_DATASETS[LOCATION_KEY]
site_label = loc["label"] # for saving

In [25]:
# # --- make any modifications
# # -----------------------------------------------------------------
# # [ MAX 10 T/HA ]
# # df["rockflx_t_ha"] = df["rockflx_t_ha"].clip(upper=10)
# # df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"].clip(upper=10e-6)
# # df["rockflx_Mt"] = np.nan

# # [ MAX 5 T/HA ]
# df["rockflx_t_ha"] = df["rockflx_t_ha"].clip(upper=5)
# df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"].clip(upper=5e-6)
# df["rockflx_Mt"] = np.nan

# # [ ROCK AMOUNT MULTIPLIER ]
# rock_multiplier = 2
# df["rockflx_t_ha"] = df["rockflx_t_ha"] * rock_multiplier
# df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"] * rock_multiplier
# df["rockflx_Mt"] = np.nan
# # -----------------------------------------------------------------
# df

In [26]:
# ---------------------------------------
# [ set up function inputs ]
t_add = 0.1
taudust = 0.05
output_filename = "Dust_temp.in"
dryrun = False
# ---------------------------------------
if REPEAT_CLIM_FOR_CROP_AREA:
        for frac in crop_area_frac_arr:
                #
                RUN_SUFFIX = f"{frac}-cropFraction"
                #
                # --- read in data
                deg_resolution = 1
                datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
                subdir = f'one-cell-per-region/{RUNNAME}/{RUN_SUFFIX}'
                fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
                fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

                # dataframe 
                df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
                all_regions = df['region_nospaces'].unique()

                # --- create output
                climpath_main = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/clim/one-cell-per-region_1deg/{RUNNAME}/{RUN_SUFFIX}"
                for this_region in all_regions:
                        print(this_region)
                        # pull out this region 
                        dfreg = df[df['region'] == this_region].copy()
                        # create output dir
                        outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
                        climpath = outdir # read and write to same dir

                        # create Dust_temp.in file
                        outdict = shf.create_dust_input_from_gcam_df(
                                outdir,
                                climpath,
                                df = dfreg,
                                t_add = t_add,
                                taudust = taudust,
                                output_filename = "Dust_temp.in",
                                dryrun = dryrun,
                        )

elif REPEAT_CLIM_FOR_APP_RATE:
        for apprate in app_rate_arr:
                #
                RUN_SUFFIX = f"{apprate}-tHaYr-appRate"
                #
                # --- read in data
                deg_resolution = 1
                datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
                subdir = f'one-cell-per-region/{RUNNAME}/{RUN_SUFFIX}'
                fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
                fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

                # dataframe 
                df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
                all_regions = df['region_nospaces'].unique()

                # --- create output
                climpath_main = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/clim/one-cell-per-region_1deg/{site_label}/{RUNNAME}/{RUN_SUFFIX}"
                for this_region in all_regions:
                        print(this_region)
                        # pull out this region 
                        dfreg = df[df['region'] == this_region].copy()
                        # create output dir
                        outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
                        climpath = outdir # read and write to same dir

                        # create Dust_temp.in file
                        outdict = shf.create_dust_input_from_gcam_df(
                                outdir,
                                climpath,
                                df = dfreg,
                                t_add = t_add,
                                taudust = taudust,
                                output_filename = "Dust_temp.in",
                                dryrun = dryrun,
                        )

else:
        # --- read in data
        deg_resolution = 1
        datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
        subdir = f'one-cell-per-region/{RUNNAME}/{RUN_SUFFIX}'
        fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
        fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

        # dataframe 
        df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
        all_regions = df['region_nospaces'].unique()
        #
        for this_region in all_regions:
                print(this_region)
                # pull out this region 
                dfreg = df[df['region'] == this_region].copy()
                # create output dir
                outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
                climpath = outdir # read and write to same dir

                # create Dust_temp.in file
                outdict = shf.create_dust_input_from_gcam_df(
                        outdir,
                        climpath,
                        df = dfreg,
                        t_add = t_add,
                        taudust = taudust,
                        output_filename = "Dust_temp.in",
                        dryrun = dryrun,
                )


USA
Africa_Eastern
Africa_Northern
Africa_Southern
Africa_Western
Australia_NZ
Brazil
Canada
Central_America_and_Caribbean
Central_Asia
China
EU-12
EU-15
Europe_Eastern
Europe_Non_EU
European_Free_Trade_Association
India
Indonesia
Japan
Mexico
Middle_East
Pakistan
Russia
South_Africa
South_America_Northern
South_America_Southern
South_Asia
South_Korea
Southeast_Asia
Taiwan
Argentina
Colombia
USA
Africa_Eastern
Africa_Northern
Africa_Southern
Africa_Western
Australia_NZ
Brazil
Canada
Central_America_and_Caribbean
Central_Asia
China
EU-12
EU-15
Europe_Eastern
Europe_Non_EU
European_Free_Trade_Association
India
Indonesia
Japan
Mexico
Middle_East
Pakistan
Russia
South_Africa
South_America_Northern
South_America_Southern
South_Asia
South_Korea
Southeast_Asia
Taiwan
Argentina
Colombia
USA
Africa_Eastern
Africa_Northern
Africa_Southern
Africa_Western
Australia_NZ
Brazil
Canada
Central_America_and_Caribbean
Central_Asia
China
EU-12
EU-15
Europe_Eastern
Europe_Non_EU
European_Free_Trade_Associa

In [20]:
# plt.plot(outdict['time_yr'], outdict['cumrate_gm2yr'])
# plt.plot(dfreg['time'], dfreg['rockflx_t_ha'].cumsum()*100)
# plt.plot(outdict['time_yr'], outdict['rate_gm2yr'])
#plt.xlim(70,80)
#plt.ylim(7e4,8.7e4)

In [ ]:
# ---
dfreg

,reg_id,region,time,rockflx_t_ha,rockflx_Mt_ha,rockflx_Mt,lat,lon,gcam_gross_cdr_mtCO2,cropland_area_required_ha,cropland_fraction_required,cropland_fraction_effective,effective_rate_t_ha,region_nospaces,rockflx_g_m2,time_yr,year
2511,32,Colombia,2020,50.0,0.00005,7.106142,3.5,-76.5,0.000000,0.000000e+00,0.000000,0.000000,50.0,Colombia,5000.0,0,0
2512,32,Colombia,2021,50.0,0.00005,7.106142,3.5,-76.5,0.216418,1.442784e+04,0.003059,0.003059,50.0,Colombia,5000.0,1,1
2513,32,Colombia,2022,50.0,0.00005,7.106142,3.5,-76.5,0.432835,2.885569e+04,0.006117,0.006117,50.0,Colombia,5000.0,2,2
2514,32,Colombia,2023,50.0,0.00005,7.106142,3.5,-76.5,0.649253,4.328353e+04,0.009176,0.009176,50.0,Colombia,5000.0,3,3
2515,32,Colombia,2024,50.0,0.00005,7.106142,3.5,-76.5,0.865671,5.771138e+04,0.012234,0.012234,50.0,Colombia,5000.0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2587,32,Colombia,2096,50.0,0.00005,7.106142,3.5,-76.5,27.597455,1.839830e+06,0.390020,0.390020,50.0,Colombia,5000.0,76,76
2588,32,Colombia,2097,50.0,0.00005,7.106142,3.5,-76.5,27.597974,1.839865e+06,0.390027,0.390027,50.0,Colombia,5000.0,77,77
2589,32,Colombia,2098,50.0,0.00005,7.106142,3.5,-76.5,27.598494,1.839900e+06,0.390035,0.390035,50.0,Colombia,5000.0,78,78
2590,32,Colombia,2099,50.0,0.00005,7.106142,3.5,-76.5,27.599014,1.839934e+06,0.390042,0.390042,50.0,Colombia,5000.0,79,79
